In [1]:
# Célula 1 - Instalar dependências e baixar dataset

import kagglehub
from pathlib import Path

data_path_str = kagglehub.dataset_download("muhammetzahitaydn/hardhat-vest-dataset-v3")
DATA_ROOT = Path(data_path_str)

print("Dataset baixado em:", DATA_ROOT)
print("Conteúdo:", list(DATA_ROOT.iterdir()))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset baixado em: /Users/lucasmarzochi/.cache/kagglehub/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3/versions/2
Conteúdo: [PosixPath('/Users/lucasmarzochi/.cache/kagglehub/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3/versions/2/images'), PosixPath('/Users/lucasmarzochi/.cache/kagglehub/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3/versions/2/labels')]


In [2]:
# Célula 2 - Imports de modelagem

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F

import matplotlib.pyplot as plt

from pathlib import Path

from PIL import Image  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IMAGES_DIR = DATA_ROOT / "images"
LABELS_DIR = DATA_ROOT / "labels"

CLASS_NAMES = {
    0: "Head",
    1: "Helmet",
    2: "Vest"
}
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 background


Device: cpu


In [3]:
# Célula 3 - Definição da classe Dataset para detecção

class HardHatDetectionDataset(Dataset):
    def __init__(self, split="train", transforms=None):
        super().__init__()
        self.split = split
        self.transforms = transforms

        self.images_dir = IMAGES_DIR / split
        self.labels_dir = LABELS_DIR / split

        self.image_paths = sorted(list(self.images_dir.glob("*.jpg")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_path = self.labels_dir / (img_path.stem + ".txt")

        # Carregar imagem
        img = Image.open(img_path).convert("RGB")
        w_img, h_img = img.size

        # Carregar boxes
        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    class_id = int(parts[0])
                    xc, yc, w_rel, h_rel = map(float, parts[1:])

                    bw = w_rel * w_img
                    bh = h_rel * h_img
                    bx = (xc * w_img) - bw / 2
                    by = (yc * h_img) - bh / 2
                    x_min = bx
                    y_min = by
                    x_max = bx + bw
                    y_max = by + bh

                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(class_id + 1)  # +1 por causa do background=0

        if len(boxes) == 0:
            # Se alguma imagem não tiver box, adiciona um box "falso" bem pequeno e label background?
            # Melhor: retornar tensor vazio, o FasterRCNN suporta.
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        # Transforms básicos: ToTensor + (opcional) flips
        img = F.to_tensor(img)

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target


In [4]:
# Célula 4 - DataLoaders

from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = HardHatDetectionDataset(split="train")
val_dataset   = HardHatDetectionDataset(split="val")
test_dataset  = HardHatDetectionDataset(split="test")

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,      # <<< AQUI: 0 workers (sem multiprocessing)
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,      # <<< idem
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,      # <<< idem
    collate_fn=collate_fn
)


In [5]:
# Célula 5 - Definir o modelo de detecção (Faster R-CNN)

from torchvision.models.detection.faster_rcnn import fasterrcnn_resnet50_fpn, FastRCNNPredictor

# NÃO baixa nada da internet, cria o modelo "do zero"
model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)

# Ajusta o head para o nº de classes (background + 3 EPIs)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model.to(device)

# Otimizador
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=1e-4, weight_decay=1e-4)



In [ ]:
# Célula 6 - Loop de treino

import time

num_epochs = 10

history = {
    "epoch": [],
    "train_loss": []
}

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    start_time = time.time()

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        num_batches += 1

    avg_loss = epoch_loss / max(num_batches, 1)
    history["epoch"].append(epoch + 1)
    history["train_loss"].append(avg_loss)

    elapsed = time.time() - start_time
    print(f"[Época {epoch+1}/{num_epochs}] Loss médio: {avg_loss:.4f} - Tempo: {elapsed:.1f}s")


In [ ]:
# Célula 7 - Gráfico de perda (loss) por época

plt.figure(figsize=(6,4))
plt.plot(history["epoch"], history["train_loss"], marker="o")
plt.xlabel("Época")
plt.ylabel("Loss médio (treino)")
plt.title("Evolução do loss de treino por época")
plt.grid(True)
plt.show()


In [ ]:
# Célula 8 - Avaliação: acurácia, F1-score e matriz de confusão

from collections import defaultdict
import numpy as np

def evaluate_presence_per_image(model, data_loader, device, score_thresh=0.5):
    model.eval()

    num_classes = len(CLASS_NAMES)
    class_ids = list(range(1, num_classes+1))  # 1..3 (ignorando background 0)

    # Contadores TP/FP/FN/TN por classe
    TP = np.zeros(num_classes, dtype=np.int64)
    FP = np.zeros(num_classes, dtype=np.int64)
    FN = np.zeros(num_classes, dtype=np.int64)
    TN = np.zeros(num_classes, dtype=np.int64)

    # Matriz de confusão de "classe dominante" por imagem
    conf_mat = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                # GT: quais classes aparecem na imagem?
                gt_labels = target["labels"].cpu().numpy()  # 1..3
                gt_present = np.zeros(num_classes, dtype=int)
                for c in gt_labels:
                    if c == 0:
                        continue
                    gt_present[c-1] = 1

                # Pred: quais classes o modelo detectou com score >= limiar?
                pred_labels = output["labels"].cpu().numpy()
                pred_scores = output["scores"].cpu().numpy()
                keep = pred_scores >= score_thresh
                pred_labels = pred_labels[keep]

                pred_present = np.zeros(num_classes, dtype=int)
                for c in pred_labels:
                    if c == 0:
                        continue
                    pred_present[c-1] = 1

                # Atualizar TP/FP/FN/TN por classe
                for i in range(num_classes):
                    if gt_present[i] == 1 and pred_present[i] == 1:
                        TP[i] += 1
                    elif gt_present[i] == 0 and pred_present[i] == 1:
                        FP[i] += 1
                    elif gt_present[i] == 1 and pred_present[i] == 0:
                        FN[i] += 1
                    else:
                        TN[i] += 1

                # Matriz de confusão com classe dominante da imagem
                # (classe GT = mais frequente na imagem; predita = classe
                # com maior score entre as detecções)
                if len(gt_labels) > 0:
                    # classe verdadeira dominante
                    gt_counts = np.zeros(num_classes, dtype=int)
                    for c in gt_labels:
                        if c == 0: 
                            continue
                        gt_counts[c-1] += 1
                    gt_main = np.argmax(gt_counts)
                else:
                    # sem GT -> considera como "sem classe" (poderia ignorar)
                    gt_main = None

                if len(pred_labels) > 0:
                    # pegar a classe da detecção com maior score
                    best_idx = np.argmax(pred_scores[keep])
                    pred_main = pred_labels[best_idx] - 1
                else:
                    pred_main = None

                if gt_main is not None and pred_main is not None:
                    conf_mat[gt_main, pred_main] += 1

    # Métricas por classe
    precision = TP / (TP + FP + 1e-6)
    recall    = TP / (TP + FN + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    accuracy  = (TP + TN) / (TP + TN + FP + FN + 1e-6)

    metrics = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "confusion_matrix": conf_mat
    }

    return metrics


In [ ]:
# Célula 9 - Executar avaliação e imprimir resultados

metrics = evaluate_presence_per_image(model, test_loader, device, score_thresh=0.5)

class_list = list(CLASS_NAMES.values())

print("Métricas por classe (presença de classe por imagem):\n")
for i, cls_name in enumerate(class_list):
    print(f"Classe: {cls_name}")
    print(f"  Precision: {metrics['precision'][i]:.4f}")
    print(f"  Recall   : {metrics['recall'][i]:.4f}")
    print(f"  F1-score : {metrics['f1'][i]:.4f}")
    print(f"  Accuracy : {metrics['accuracy'][i]:.4f}")
    print()

# Acurácia média
acc_media = metrics["accuracy"].mean()
print(f"Acurácia média (média das classes): {acc_media:.4f}")


In [ ]:
# Célula 10 - Plotar matriz de confusão

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

conf_mat = metrics["confusion_matrix"]
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(conf_mat, annot=True, fmt="d",
            xticklabels=class_list,
            yticklabels=class_list,
            cmap="Blues",
            ax=ax)
ax.set_xlabel("Classe predita (imagem)")
ax.set_ylabel("Classe verdadeira (imagem)")
ax.set_title("Matriz de confusão - classe dominante por imagem")
plt.tight_layout()
plt.show()
